In [ ]:
# Install astronomy helpers
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sunpy[all]', 'astropy', 'pyerfa'],
               capture_output=True)
print("✓ Dependencies ready")

In [ ]:
import os, warnings, glob, time
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output, display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from astropy.io import fits
from astropy.time import Time

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

SEED = 2024
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f"✓ Device  : {device}")
print(f"✓ GPUs    : {NUM_GPUS}")
if device.type == 'cuda':
    for i in range(NUM_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}  : {p.name}  ({p.total_memory/1e9:.1f} GB)")
print(f"✓ PyTorch : {torch.__version__}")

In [ ]:
C_THRESHOLD = 29.0          # C-class ≥ 29  counts/s
M_THRESHOLD = 131.4         # M-class ≥ 131.4 counts/s
X_THRESHOLD = 1_000.0       # X-class ≥ 1000 counts/s

CADENCE_S     = 10          # seconds between samples
HISTORY_STEPS = 120         # 20-min input  (120 × 10 s)
FORECAST_STEPS = 60         # 10-min output (60  × 10 s)

HIDDEN_DIM  = 192           # channel width throughout
TCN_BLOCKS  = 6             # dilated blocks — dilations 1,2,4,8,16,32
DROPOUT     = 0.15

BATCH_SIZE   = 384          # per GPU; effective = BATCH_SIZE × NUM_GPUS
EPOCHS       = 50
LR           = 3e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP    = 1.0

W_REG    = 1.00   # Huber regression on full sequence
W_GRAD   = 0.40   # temporal-gradient penalty (anti-flat-line)
W_CLASS  = 0.15   # cross-entropy for flare class
W_PEAK   = 0.25   # Huber on predicted vs actual peak

SOLEXS_DIR = 'PATH_TO_SOLEXS_DATA'
HELIOS_DIR = 'PATH_TO_HELIOS_DATA'

INPUT_DIM  = 10             # number of input features

print("✓ Configuration loaded")
print(f"  History  : {HISTORY_STEPS * CADENCE_S // 60} min  ({HISTORY_STEPS} steps)")
print(f"  Forecast : {FORECAST_STEPS * CADENCE_S // 60} min ({FORECAST_STEPS} steps)")
print(f"  Batch    : {BATCH_SIZE} × {max(1, NUM_GPUS)} GPU(s) = {BATCH_SIZE * max(1, NUM_GPUS)} effective")

In [ ]:
def load_fits_table(path):
    """Return the first binary-table extension of a FITS file as a DataFrame."""
    with fits.open(path) as hdul:
        for ext in hdul:
            if not ext.is_image and ext.data is not None:
                df = pd.DataFrame(ext.data)
                df.columns = [c.strip().upper() for c in df.columns]
                return df
    raise ValueError(f"No table extension in {path}")

def parse_solexs_file(path):
    df = load_fits_table(path)
    tcol = 'TIME' if 'TIME' in df.columns else df.columns[0]
    rcol = next((c for c in ('COUNTS', 'RATE', 'CTR') if c in df.columns), df.columns[1])
    out = pd.DataFrame({
        'time'        : pd.to_datetime(df[tcol], unit='s').dt.tz_localize(None),
        'solexs_rate' : df[rcol].astype(float).clip(lower=0)
    })
    return out.sort_values('time').reset_index(drop=True)

def parse_helios_file(path):
    df = load_fits_table(path)
    if 'ISOT' in df.columns:
        times = pd.to_datetime(df['ISOT']).dt.tz_localize(None)
    elif 'MJD' in df.columns:
        times = pd.to_datetime(Time(df['MJD'].values, format='mjd').datetime).dt.tz_localize(None)
    elif 'TIME' in df.columns:
        times = pd.to_datetime(df['TIME'], unit='s').dt.tz_localize(None)
    else:
        times = pd.to_datetime(df[df.columns[0]]).dt.tz_localize(None)

    rcol = next((c for c in ('CTR', 'COUNTS', 'RATE', 'COUNT_RATE') if c in df.columns), df.columns[1])
    out = pd.DataFrame({
        'time'        : times,
        'helios_rate' : df[rcol].astype(float).clip(lower=0)
    })
    return out.sort_values('time').reset_index(drop=True)

def load_payload_data(solexs_dir, helios_dir, cadence='10S'):
    """
    Load, resample and merge SoLEXS + HEL1OS light curves onto a uniform grid.
    Returns a DataFrame with columns: solexs_rate, helios_rate.
    """
    sx_files = glob.glob(os.path.join(solexs_dir, '*'))
    hx_files = glob.glob(os.path.join(helios_dir, '*'))
    print(f"  SoLEXS files : {len(sx_files)}")
    print(f"  HEL1OS files : {len(hx_files)}")

    sx_dfs, hx_dfs = [], []
    for f in sx_files:
        try:   sx_dfs.append(parse_solexs_file(f))
        except Exception as e: print(f"  [skip SoLEXS] {os.path.basename(f)}: {e}")
    for f in hx_files:
        try:   hx_dfs.append(parse_helios_file(f))
        except Exception as e: print(f"  [skip HEL1OS] {os.path.basename(f)}: {e}")

    if not sx_dfs: raise FileNotFoundError("No SoLEXS files parsed.")
    if not hx_dfs: raise FileNotFoundError("No HEL1OS files parsed.")

    df_sx = (pd.concat(sx_dfs).drop_duplicates('time')
               .set_index('time').resample(cadence).mean()
               .interpolate('linear').tz_localize(None))
    df_hx = (pd.concat(hx_dfs).drop_duplicates('time')
               .set_index('time').resample(cadence).mean()
               .interpolate('linear').tz_localize(None))

    df = (df_sx.join(df_hx, how='outer')
               .interpolate('linear').fillna(0)
               .clip(lower=0))
    print(f"  Merged span  : {df.index[0]} → {df.index[-1]}  ({len(df):,} rows)")
    return df

print("✓ FITS loaders defined")

In [ ]:
def _inject_flare(signal, n, t0, amplitude, rise_steps, decay_steps):
    """Solar flare profile: Gaussian rise + exponential decay."""
    peak = t0 + rise_steps
    # Rise
    for t in range(max(0, t0), min(n, peak)):
        frac = (t - t0) / max(rise_steps, 1)
        signal[t] += amplitude * frac ** 2          # quadratic brightening
    # Decay
    tau = max(decay_steps / 2.5, 1)
    for t in range(min(n, peak), min(n, peak + decay_steps * 2)):
        signal[t] += amplitude * np.exp(-(t - peak) / tau)


def create_synthetic_solar_data(n_days=60, cadence_s=10, seed=SEED):
    """
    Generate a realistic SoLEXS + HEL1OS multi-day time-series.
    Flare rates approximate solar-maximum activity levels.
    """
    rng = np.random.default_rng(seed)
    n   = int(n_days * 86_400 / cadence_s)           # total steps
    t   = np.arange(n) * cadence_s                   # seconds axis

    # Slow drift (≈ 11-min p-mode + 1-hour slow oscillation)
    bg = ( 6.0
           + 2.0 * np.sin(2 * np.pi * t / 660)        # 11-min
           + 1.5 * np.sin(2 * np.pi * t / 3600)        # 1-hour
           + rng.normal(0, 0.4, n) )                   # photon noise
    bg = np.clip(bg, 0.5, None)

    sx_flare = np.zeros(n)
    hx_flare = np.zeros(n)

    def add_flares(n_events, amp_lo, amp_hi, rise_lo, rise_hi, decay_lo, decay_hi):
        for _ in range(n_events):
            t0  = rng.integers(200, n - 3_000)
            amp = rng.uniform(amp_lo, amp_hi)
            rise  = int(rng.uniform(rise_lo, rise_hi)   * 60 / cadence_s)
            decay = int(rng.uniform(decay_lo, decay_hi) * 60 / cadence_s)
            _inject_flare(sx_flare, n, t0, amp, rise, decay)
            # HEL1OS responds mainly to non-thermal impulsive phase
            hx_amp = amp * rng.uniform(0.25, 0.55)
            _inject_flare(hx_flare, n, t0 + rise // 2, hx_amp,
                          max(1, rise // 3), decay // 2)

    # C-class: ~8 / day, amplitude 25–100 above background
    add_flares(n_days * 8,  amp_lo=25,   amp_hi=100,
               rise_lo=2,  rise_hi=8,   decay_lo=8,  decay_hi=35)
    # M-class: ~1.5 / day
    add_flares(int(n_days * 1.5), amp_lo=100, amp_hi=1_100,
               rise_lo=4, rise_hi=15,  decay_lo=15, decay_hi=70)
    # X-class: ~0.2 / day
    add_flares(max(2, n_days // 5), amp_lo=1_100, amp_hi=7_000,
               rise_lo=5, rise_hi=20, decay_lo=30, decay_hi=120)

    solexs_rate = np.clip(bg + sx_flare, 0.1, None)
    helios_rate = np.clip(
        rng.normal(0, 0.8, n) + hx_flare * 0.8, 0, None
    )

    dates = pd.date_range('2024-01-01', periods=n, freq=f'{cadence_s}S')
    df = pd.DataFrame({'solexs_rate': solexs_rate, 'helios_rate': helios_rate},
                      index=dates)
    print(f"✓ Synthetic data: {n_days} days  |  {n:,} steps  |  cadence {cadence_s}s")

    # Summary
    tot = solexs_rate
    c_ev = int(np.sum(tot > C_THRESHOLD) * cadence_s / 300)   # rough event count
    m_ev = int(np.sum(tot > M_THRESHOLD) * cadence_s / 600)
    x_ev = int(np.sum(tot > X_THRESHOLD) * cadence_s / 1200)
    print(f"  ≥C : ~{c_ev} events  |  ≥M : ~{m_ev}  |  ≥X : ~{x_ev}")
    print(f"  SoLEXS  min={solexs_rate.min():.1f}  mean={solexs_rate.mean():.1f}"
          f"  max={solexs_rate.max():.1f}  counts/s")
    return df

print("✓ Synthetic data generator defined")

In [ ]:
def build_features(df):
    """
    Compute all input features in log1p space.
    Returns a DataFrame with all feature columns + 'target'.
    """
    d = df.copy()

    d['sx_log'] = np.log1p(d['solexs_rate'].values)   # ~[0, 9] for full range
    d['hx_log'] = np.log1p(d['helios_rate'].values)

    for w, tag in [(12, '2m'), (30, '5m')]:   # 12×10s=2min, 30×10s=5min
        d[f'sx_rmax_{tag}']  = d['sx_log'].rolling(w, min_periods=1).max()
        d[f'sx_rmean_{tag}'] = d['sx_log'].rolling(w, min_periods=1).mean()
        d[f'hx_rmax_{tag}']  = d['hx_log'].rolling(w, min_periods=1).max()
        d[f'hx_rmean_{tag}'] = d['hx_log'].rolling(w, min_periods=1).mean()

    d['sx_delta'] = d['sx_log'].diff().fillna(0)
    d['hx_delta'] = d['hx_log'].diff().fillna(0)

    d['target'] = d['sx_log']

    return d.fillna(0)

# Feature column order (must match INPUT_DIM=10)
FEATURE_COLS = [
    'sx_log', 'hx_log',
    'sx_rmax_2m', 'sx_rmean_2m',
    'sx_rmax_5m', 'sx_rmean_5m',
    'hx_rmax_2m', 'hx_rmean_2m',
    'sx_delta',   'hx_delta',
]
assert len(FEATURE_COLS) == INPUT_DIM


class FeatureNormalizer:
    def __init__(self):
        self.mean_  = None
        self.std_   = None

    def fit(self, X):                         # X: [N, D]
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0) + 1e-8

    def transform(self, X):
        return (X - self.mean_) / self.std_

    def inverse_target(self, y_norm):
        """y_norm → log1p(counts/s) → physical counts/s"""
        y_log1p = y_norm * self.std_[0] + self.mean_[0]
        return np.expm1(np.clip(y_log1p, 0, None))

    # Thresholds in normalised space for classification labels
    def log1p_thresholds(self):
        """C, M, X thresholds in normalised log1p space."""
        raw = np.array([np.log1p(C_THRESHOLD),
                        np.log1p(M_THRESHOLD),
                        np.log1p(X_THRESHOLD)])
        return (raw - self.mean_[0]) / self.std_[0]


print("✓ Preprocessing helpers defined")
print(f"  Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")

In [ ]:
class SolarFlareDataset(Dataset):

    def __init__(self, X_norm, y_norm, history=HISTORY_STEPS,
                 forecast=FORECAST_STEPS, thresholds_norm=None):
        self.X  = torch.tensor(X_norm, dtype=torch.float32)
        self.y  = torch.tensor(y_norm, dtype=torch.float32)
        self.H  = history
        self.F  = forecast
        self.N  = len(X_norm) - history - forecast + 1
        self.thrs = thresholds_norm   # (t_C, t_M, t_X) in norm space

        # Pre-compute per-sample weights and labels
        weights = np.ones(self.N, dtype=np.float32)
        labels  = np.zeros(self.N, dtype=np.int64)
        if self.thrs is not None:
            t_C, t_M, t_X = self.thrs
            for i in range(self.N):
                peak = y_norm[i + history : i + history + forecast].max()
                if   peak >= t_X: weights[i], labels[i] = 60.0, 3
                elif peak >= t_M: weights[i], labels[i] = 20.0, 2
                elif peak >= t_C: weights[i], labels[i] =  4.0, 1
        self.weights = torch.tensor(weights, dtype=torch.float32)
        self.labels  = torch.tensor(labels,  dtype=torch.long)

    def __len__(self): return self.N

    def __getitem__(self, idx):
        x  = self.X[idx : idx + self.H]
        y  = self.y[idx + self.H : idx + self.H + self.F]
        return x, y, self.weights[idx], self.labels[idx]


def make_loaders(df_feat, normalizer, val_frac=0.15, test_frac=0.05):
    """
    Chronological split → DataLoaders.
    Returns train_loader, val_loader, test_loader, split_indices.
    """
    X_raw = df_feat[FEATURE_COLS].values.astype(np.float32)
    y_raw = df_feat['target'].values.astype(np.float32)

    N = len(X_raw)
    n_test = max(HISTORY_STEPS + FORECAST_STEPS + 1, int(N * test_frac))
    n_val  = max(HISTORY_STEPS + FORECAST_STEPS + 1, int(N * val_frac))
    n_train = N - n_val - n_test

    X_tr, y_tr = X_raw[:n_train],             y_raw[:n_train]
    X_vl, y_vl = X_raw[n_train:n_train+n_val],y_raw[n_train:n_train+n_val]
    X_te, y_te = X_raw[n_train+n_val:],        y_raw[n_train+n_val:]

    normalizer.fit(X_tr)
    X_tr = normalizer.transform(X_tr)
    X_vl = normalizer.transform(X_vl)
    X_te = normalizer.transform(X_te)

    # Also normalise target using feature-0 statistics
    y_tr_n = (y_tr - normalizer.mean_[0]) / normalizer.std_[0]
    y_vl_n = (y_vl - normalizer.mean_[0]) / normalizer.std_[0]
    y_te_n = (y_te - normalizer.mean_[0]) / normalizer.std_[0]

    thrs = normalizer.log1p_thresholds()

    ds_tr = SolarFlareDataset(X_tr, y_tr_n, thresholds_norm=thrs)
    ds_vl = SolarFlareDataset(X_vl, y_vl_n, thresholds_norm=thrs)
    ds_te = SolarFlareDataset(X_te, y_te_n, thresholds_norm=thrs)

    # Weighted sampler for training (oversample flare sequences)
    sampler = torch.utils.data.WeightedRandomSampler(
        ds_tr.weights, len(ds_tr), replacement=True)

    tr_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=4, pin_memory=True, drop_last=True)
    vl_loader = DataLoader(ds_vl, batch_size=BATCH_SIZE * 2, shuffle=False,
                           num_workers=4, pin_memory=True)
    te_loader = DataLoader(ds_te, batch_size=BATCH_SIZE * 2, shuffle=False,
                           num_workers=4, pin_memory=True)

    print(f"✓ Dataset  train={len(ds_tr):,}  val={len(ds_vl):,}  test={len(ds_te):,}")
    print(f"  Thresholds (norm) : C={thrs[0]:.3f}  M={thrs[1]:.3f}  X={thrs[2]:.3f}")
    return tr_loader, vl_loader, te_loader, (n_train, n_val, n_test), normalizer


print("✓ Dataset class defined")

In [ ]:
class CausalConv1d(nn.Module):
    """Causal (left-padded) dilated 1D convolution."""
    def __init__(self, in_ch, out_ch, kernel, dilation):
        super().__init__()
        pad = (kernel - 1) * dilation          # left padding only
        self.pad  = pad
        self.conv = nn.Conv1d(in_ch, out_ch, kernel,
                              dilation=dilation, padding=pad)

    def forward(self, x):                      # x: [B, C, T]
        return self.conv(x)[:, :, :x.shape[-1]]


class TCNBlock(nn.Module):
    """Residual dilated-causal-convolution block."""
    def __init__(self, ch, kernel=3, dilation=1, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(ch, ch, kernel, dilation),
            nn.GroupNorm(min(8, ch), ch),
            nn.GELU(),
            nn.Dropout(dropout),
            CausalConv1d(ch, ch, kernel, dilation),
            nn.GroupNorm(min(8, ch), ch),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class SolarFlareTCN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden=HIDDEN_DIM,
                 n_blocks=TCN_BLOCKS, forecast_steps=FORECAST_STEPS, dropout=DROPOUT):
        super().__init__()
        self.forecast_steps = forecast_steps

        self.in_proj = nn.Sequential(
            nn.Conv1d(input_dim, hidden, kernel_size=1),
            nn.GroupNorm(min(8, hidden), hidden),
            nn.GELU(),
        )

        dilations = [2 ** i for i in range(n_blocks)]
        self.tcn  = nn.Sequential(*[
            TCNBlock(hidden, kernel=3, dilation=d, dropout=dropout)
            for d in dilations
        ])

        # query = last timestep; keys/values = full sequence
        self.attn_proj = nn.Linear(hidden, hidden)
        self.attn_w    = nn.Linear(hidden, 1)

        self.fc_forecast = nn.Sequential(
            nn.Linear(hidden * 2, hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden, forecast_steps),
        )

        # Initialised small so training starts near persistence forecast
        self.skip = nn.Linear(1, forecast_steps, bias=True)
        nn.init.constant_(self.skip.weight, 1.0 / forecast_steps)
        nn.init.zeros_(self.skip.bias)

        self.fc_class = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Linear(hidden, 4),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        # Re-init skip to near-persistence
        nn.init.constant_(self.skip.weight, 1.0 / self.forecast_steps)
        nn.init.zeros_(self.skip.bias)

    def forward(self, x):
        """
        x   : [B, T, D]  — z-normalised features, feature-0 = normalised sx_log
        out : pred [B, F], class_logits [B, 4]
        """
        B, T, D = x.shape

        h = x.permute(0, 2, 1)       # [B, D, T]
        h = self.in_proj(h)           # [B, H, T]
        h = self.tcn(h)               # [B, H, T]
        h = h.permute(0, 2, 1)        # [B, T, H]

        a = torch.tanh(self.attn_proj(h))           # [B, T, H]
        w = torch.softmax(self.attn_w(a), dim=1)    # [B, T, 1]
        h_global = (w * h).sum(dim=1)               # [B, H]
        h_last   = h[:, -1, :]                      # [B, H]  last timestep
        ctx      = torch.cat([h_last, h_global], dim=-1)   # [B, 2H]

        delta = self.fc_forecast(ctx)                       # [B, F]
        last_val = x[:, -1, 0:1]                            # [B, 1]  normalised sx_log
        skip     = self.skip(last_val)                      # [B, F]
        pred     = skip + delta                             # [B, F]  final prediction

        cls = self.fc_class(ctx)                            # [B, 4]

        return pred, cls


# Quick sanity check
_m = SolarFlareTCN()
_x = torch.randn(4, HISTORY_STEPS, INPUT_DIM)
_p, _c = _m(_x)
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"✓ Model : SolarFlareTCN")
print(f"  Input  : {list(_x.shape)}  →  pred {list(_p.shape)}  cls {list(_c.shape)}")
print(f"  Params : {n_params:,}")
del _m, _x, _p, _c

In [ ]:
class AdaptiveTemporalLoss(nn.Module):
    def __init__(self, w_reg=W_REG, w_grad=W_GRAD, w_class=W_CLASS, w_peak=W_PEAK,
                 huber_beta=0.5):
        super().__init__()
        self.w_reg   = w_reg
        self.w_grad  = w_grad
        self.w_class = w_class
        self.w_peak  = w_peak
        self.hb      = huber_beta

    def forward(self, pred, target, cls_logits, cls_labels, sample_weights):
        sw = sample_weights / (sample_weights.mean() + 1e-8)   # normalise weights

        huber = F.smooth_l1_loss(pred, target, beta=self.hb, reduction='none')  # [B, F]
        reg_loss = (huber.mean(dim=1) * sw).mean()

        pred_diff   = pred[:, 1:] - pred[:, :-1]       # [B, F-1]
        target_diff = target[:, 1:] - target[:, :-1]
        grad_loss   = F.mse_loss(pred_diff, target_diff)

        class_loss = F.cross_entropy(cls_logits, cls_labels)

        pred_peak   = pred.max(dim=1).values                   # [B]
        target_peak = target.max(dim=1).values
        peak_loss   = (F.smooth_l1_loss(pred_peak, target_peak,
                                         beta=self.hb, reduction='none') * sw).mean()

        total = (self.w_reg   * reg_loss   +
                 self.w_grad  * grad_loss  +
                 self.w_class * class_loss +
                 self.w_peak  * peak_loss)

        return total, {
            'reg'  : reg_loss.item(),
            'grad' : grad_loss.item(),
            'cls'  : class_loss.item(),
            'peak' : peak_loss.item(),
        }

print("✓ AdaptiveTemporalLoss defined")

In [ ]:
def run_epoch(model, loader, criterion, optimizer, scaler, is_train, device):
    model.train() if is_train else model.eval()
    tot, n = 0.0, 0
    comp_sums = {'reg': 0., 'grad': 0., 'cls': 0., 'peak': 0.}

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y, wt, lbl in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            wt   = wt.to(device,  non_blocking=True)
            lbl  = lbl.to(device, non_blocking=True)

            with autocast():
                pred, cls = model(x)
                loss, comp = criterion(pred, y, cls, lbl, wt)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer); scaler.update()

            bs = x.size(0)
            tot += loss.item() * bs
            n   += bs
            for k in comp_sums: comp_sums[k] += comp[k] * bs

    return tot / n, {k: v / n for k, v in comp_sums.items()}


def train_model(model, tr_loader, vl_loader, criterion, epochs=EPOCHS):
    optim  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs, eta_min=LR * 0.05)
    scaler = GradScaler()

    best_val, patience, max_patience = 1e9, 0, 12
    best_state = None
    history    = {'train': [], 'val': [], 'lr': []}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    plt.subplots_adjust(hspace=0.4, wspace=0.4)

    start_time = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_comp = run_epoch(model, tr_loader, criterion, optim, scaler, True,  device)
        vl_loss, vl_comp = run_epoch(model, vl_loader, criterion, optim, scaler, False, device)
        sched.step()

        lr_now = optim.param_groups[0]['lr']
        history['train'].append(tr_loss); history['val'].append(vl_loss); history['lr'].append(lr_now)

        if vl_loss < best_val:
            best_val   = vl_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
        else:
            patience  += 1

        elapsed = time.time() - start_time
        print(f"Epoch {epoch:3d}/{epochs} | "
              f"train={tr_loss:.4f}  val={vl_loss:.4f}  "
              f"[reg={tr_comp['reg']:.3f} grad={tr_comp['grad']:.3f} "
              f"cls={tr_comp['cls']:.3f} peak={tr_comp['peak']:.3f}]  "
              f"lr={lr_now:.2e}  patience={patience}  {elapsed:.0f}s")

        # Live loss plot every 5 epochs
        if epoch % 5 == 0 or epoch == epochs:
            ep_ax = list(range(1, epoch + 1))
            for ax in axes: ax.cla()

            axes[0].plot(ep_ax, history['train'], label='Train', color='royalblue')
            axes[0].plot(ep_ax, history['val'],   label='Val',   color='tomato')
            axes[0].set(title='Total Loss', xlabel='Epoch', ylabel='Loss')
            axes[0].legend(); axes[0].grid(alpha=0.3)

            axes[1].semilogy(ep_ax, history['lr'], color='seagreen')
            axes[1].set(title='Learning Rate', xlabel='Epoch', ylabel='LR')
            axes[1].grid(alpha=0.3)

            # Component breakdown (last epoch)
            comps   = ['reg', 'grad', 'cls', 'peak']
            weights = [W_REG, W_GRAD, W_CLASS, W_PEAK]
            vals    = [tr_comp[k] * w for k, w in zip(comps, weights)]
            axes[2].bar(comps, vals, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
            axes[2].set(title=f'Loss Components (Epoch {epoch})', ylabel='Weighted value')
            axes[2].grid(alpha=0.3, axis='y')

            plt.suptitle(f'Training Progress — Epoch {epoch}/{epochs}', fontweight='bold')
            clear_output(wait=True); display(fig); plt.pause(0.01)

        if patience >= max_patience:
            print(f"  ↳ Early stopping at epoch {epoch}  (best val={best_val:.4f})")
            break

    plt.close(fig)
    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"\n✓ Training complete  |  best_val={best_val:.4f}")
    return model, history


print("✓ Training loop defined")

In [ ]:
def classify_alert(peak_physical):
    if   peak_physical >= X_THRESHOLD: return 'X-Class (Severe)',   '#8B0000'
    elif peak_physical >= M_THRESHOLD: return 'M-Class (Moderate)', '#FF4500'
    elif peak_physical >= C_THRESHOLD: return 'C-Class (Minor)',    '#FFA500'
    else:                              return 'Quiet (No Impact)',   '#2196F3'


def _log_r2(actual, pred):
    ss_res = np.sum((actual - pred) ** 2)
    ss_tot = np.sum((actual - actual.mean()) ** 2)
    return float(1 - ss_res / (ss_tot + 1e-12))


@torch.no_grad()
def plot_forecast_grid(model, dataset, normalizer, n_samples=6,
                       cadence_s=CADENCE_S, title='Forecast vs Actual'):
    """
    Plot a grid of inference samples drawn from the dataset, in PHYSICAL counts/s.
    Samples are spread across the dataset to show quiet, C, M activity levels.
    """
    model.eval()
    N = len(dataset)
    # Spread indices: quartiles + extreme (high-flux) samples
    base_idxs = np.linspace(0, N - 1, n_samples, dtype=int).tolist()

    # Override some with high-flux samples (flare events)
    peaks = [dataset[i][1].max().item() for i in range(N)]
    sorted_idx = np.argsort(peaks)[::-1]
    for k in range(min(2, n_samples // 2)):
        base_idxs[-(k+1)] = int(sorted_idx[k])

    ncols, nrows = 3, (n_samples + 2) // 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 4))
    axes = np.array(axes).flatten()
    H, F = HISTORY_STEPS, FORECAST_STEPS
    t_hist     = np.arange(H) * cadence_s
    t_forecast = np.arange(H, H + F) * cadence_s

    for ax_i, idx in enumerate(base_idxs):
        ax = axes[ax_i]
        x, y_norm, _, _ = dataset[idx]
        x_batch = x.unsqueeze(0).to(device)

        pred_norm, cls_logits = model(x_batch)
        pred_norm = pred_norm.squeeze(0).cpu().numpy()
        y_norm_np = y_norm.numpy()

        pred_phys = normalizer.inverse_target(pred_norm)
        act_phys  = normalizer.inverse_target(y_norm_np)
        hist_phys = normalizer.inverse_target(x[:, 0].numpy())   # feature-0 = sx_log

        r2 = _log_r2(y_norm_np, pred_norm)
        pred_alert, pred_color = classify_alert(pred_phys.max())
        act_alert,  act_color  = classify_alert(act_phys.max())

        ax.plot(t_hist,     hist_phys, 'b-',  lw=1.5, label='Input history')
        ax.plot(t_forecast, act_phys,  'g-',  lw=2,   label='Actual future')
        ax.plot(t_forecast, pred_phys, 'r--', lw=2,   label='Predicted future')
        ax.fill_between(t_forecast, act_phys, pred_phys,
                        alpha=0.12, color='red', label='Mismatch margin')
        ax.axvline(t_hist[-1], color='gray', ls=':', lw=1.5)
        ax.axhline(C_THRESHOLD, color='orange', ls='--', alpha=0.6, lw=1, label=f'C ({C_THRESHOLD:.0f})')
        ax.axhline(M_THRESHOLD, color='purple', ls='--', alpha=0.6, lw=1, label=f'M ({M_THRESHOLD:.0f})')
        ax.set_yscale('log'); ax.set_ylim(bottom=0.5)
        ax.set_xlabel('Time (s)'); ax.set_ylabel('SoLEXS (counts/s)')
        ax.set_title(f'Sample #{idx} | LogR²={r2:.3f}\n'
                     f'Actual: {act_alert}  Pred: {pred_alert}', fontsize=9)
        ax.legend(fontsize=7, loc='upper left'); ax.grid(True, which='both', alpha=0.3)

    for j in range(len(base_idxs), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return fig


@torch.no_grad()
def run_custom_inference(model, df_full, normalizer,
                         sample_idx=None, start_time_str=None,
                         history_steps=HISTORY_STEPS, forecast_steps=FORECAST_STEPS,
                         cadence_s=CADENCE_S):
    """
    Run inference on a specific time-window and show the alert-level comparison plot.
    df_full must already have been processed with build_features().
    """
    model.eval()
    N = len(df_full)
    total = history_steps + forecast_steps

    if start_time_str is not None:
        try:
            target_dt = pd.to_datetime(start_time_str).tz_localize(None)
            sample_idx = df_full.index.get_indexer([target_dt], method='nearest')[0]
            print(f"  Aligned '{start_time_str}' → index {sample_idx} ({df_full.index[sample_idx]})")
        except Exception as e:
            print(f"  parse error: {e}, using random index")
            sample_idx = None

    if sample_idx is None:
        sample_idx = np.random.randint(0, N - total)
        print(f"  Random index: {sample_idx}  ({df_full.index[sample_idx]})")
    else:
        sample_idx = min(max(0, sample_idx), N - total)

    window   = df_full.iloc[sample_idx : sample_idx + total]
    X_raw    = window[FEATURE_COLS].values.astype(np.float32)
    X_norm   = normalizer.transform(X_raw)

    x_in     = torch.tensor(X_norm[:history_steps]).unsqueeze(0).to(device)
    pred_norm, cls_log = model(x_in)
    pred_norm = pred_norm.squeeze(0).cpu().numpy()

    target_raw  = window['target'].values.astype(np.float32)
    target_norm = (target_raw - normalizer.mean_[0]) / normalizer.std_[0]

    hist_phys = normalizer.inverse_target(target_norm[:history_steps])
    act_phys  = normalizer.inverse_target(target_norm[history_steps:])
    pred_phys = normalizer.inverse_target(pred_norm)

    t_hist = np.arange(history_steps) * cadence_s
    t_fore = np.arange(history_steps, history_steps + forecast_steps) * cadence_s

    pred_alert, pred_color = classify_alert(pred_phys.max())
    act_alert,  act_color  = classify_alert(act_phys.max())
    r2 = _log_r2(target_norm[history_steps:], pred_norm)

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(t_hist, hist_phys, 'b-',  lw=2,   label=f'Input History ({history_steps*cadence_s//60} min)')
    ax.plot(t_fore, act_phys,  'g-',  lw=2.5, label='Actual Future Target')
    ax.plot(t_fore, pred_phys, 'r--', lw=2.5, label='Predicted Future Target')
    ax.fill_between(t_fore, act_phys, pred_phys, alpha=0.15, color='red', label='Forecast Mismatch Margin')
    ax.axvline(t_hist[-1], color='gray', ls=':', lw=1.5, label='Forecast Horizon Start')
    ax.axhline(C_THRESHOLD, color='orange', ls='--', alpha=0.7, lw=1.5,
               label=f'C-class threshold ({C_THRESHOLD:.1f})')
    ax.axhline(M_THRESHOLD, color='purple', ls='--', alpha=0.7, lw=1.5,
               label=f'M-class threshold ({M_THRESHOLD:.1f})')
    ax.set_yscale('log'); ax.set_ylim(bottom=0.5)
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_ylabel('SoLEXS Count Rate (counts/s)', fontsize=11)
    ax.set_title(
        f'Custom Inference Verification  |  Index: {sample_idx}\n'
        f'Actual Alert: {act_alert} (Peak: {act_phys.max():.1f})  |  '
        f'Predicted Alert: {pred_alert} (Peak: {pred_phys.max():.1f})  |  LogR²={r2:.3f}',
        fontsize=11, fontweight='bold'
    )
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout(); plt.show()
    return fig


print("✓ Visualisation helpers defined")

In [ ]:
@torch.no_grad()
def evaluate_classification(model, loader, normalizer, device):
    """
    Compute confusion matrix + TSS + precision/recall for each class.
    Returns a summary dict.
    """
    model.eval()
    all_pred, all_true = [], []
    thrs = normalizer.log1p_thresholds()

    for x, y, _, _ in loader:
        x = x.to(device)
        pred_norm, _ = model(x)
        pred_norm = pred_norm.cpu().numpy()
        y_np      = y.numpy()

        for i in range(len(pred_norm)):
            peak_p = pred_norm[i].max()
            peak_t = y_np[i].max()
            def cls(v):
                if   v >= thrs[2]: return 3
                elif v >= thrs[1]: return 2
                elif v >= thrs[0]: return 1
                else:              return 0
            all_pred.append(cls(peak_p))
            all_true.append(cls(peak_t))

    from collections import Counter
    class_names = ['Quiet', 'C', 'M', 'X']
    n_cls = len(class_names)
    cm = np.zeros((n_cls, n_cls), dtype=int)
    for t, p in zip(all_true, all_pred):
        cm[t][p] += 1

    print("\n── Confusion Matrix (rows=true, cols=pred) ───────────────────────────")
    header = '          ' + ' '.join(f'{c:>8}' for c in class_names)
    print(header)
    for i, row_name in enumerate(class_names):
        print(f'  {row_name:>6}  ' + ' '.join(f'{cm[i,j]:>8}' for j in range(n_cls)))

    print("\n── Per-class Metrics ────────────────────────────────────────────────")
    tss_list = []
    for c in range(n_cls):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        TPR = TP / (TP + FN + 1e-9)
        TNR = TN / (TN + FP + 1e-9)
        TSS = TPR + TNR - 1
        PRE = TP / (TP + FP + 1e-9)
        tss_list.append(TSS)
        print(f'  {class_names[c]:>6} | TPR={TPR:.3f}  TNR={TNR:.3f}  Prec={PRE:.3f}  TSS={TSS:.3f}')

    print(f"\n  Mean TSS (macro) = {np.mean(tss_list):.3f}")
    print(f"  Overall accuracy = {np.diag(cm).sum() / cm.sum():.3f}")
    return {'cm': cm, 'tss': tss_list}

print("✓ Evaluation helpers defined")

In [ ]:
print("=" * 65)
print("  SOLAR FLARE FORECASTING PIPELINE  —  Aditya-L1 / SoLEXS+HEL1OS")
print("=" * 65)

print("\n[1/6] Loading data …")
try:
    df_raw = load_payload_data(SOLEXS_DIR, HELIOS_DIR, cadence='10S')
    print("  ✓ Real Aditya-L1 data loaded")
    DATA_SOURCE = 'real'
except Exception as exc:
    print(f"  ⚠  FITS load failed ({exc})")
    print("  ↳  Using calibrated synthetic data (60 days)")
    df_raw = create_synthetic_solar_data(n_days=60)
    DATA_SOURCE = 'synthetic'

df_raw = df_raw.replace([np.inf, -np.inf], np.nan).dropna()
print(f"  Rows: {len(df_raw):,}  |  source: {DATA_SOURCE}")

print("\n[2/6] Building features …")
df_feat = build_features(df_raw)

fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)
axes[0].semilogy(df_feat.index, df_raw['solexs_rate'], lw=0.5, color='darkorange')
axes[0].axhline(C_THRESHOLD, ls='--', color='orange', lw=1, label=f'C ({C_THRESHOLD})')
axes[0].axhline(M_THRESHOLD, ls='--', color='red',    lw=1, label=f'M ({M_THRESHOLD})')
axes[0].set_ylabel('SoLEXS (counts/s)'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[0].set_title('Aditya-L1 Light Curves & Feature Overview', fontweight='bold')

axes[1].semilogy(df_feat.index, df_raw['helios_rate'].clip(lower=0.01), lw=0.5, color='mediumpurple')
axes[1].set_ylabel('HEL1OS (counts/s)'); axes[1].grid(alpha=0.3)

axes[2].plot(df_feat.index, df_feat['sx_log'], lw=0.5, color='steelblue', label='log1p(SoLEXS) = target')
axes[2].set_ylabel('log1p value'); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('Time')

plt.tight_layout(); plt.show()

print("\n[3/6] Creating datasets …")
normalizer = FeatureNormalizer()
tr_loader, vl_loader, te_loader, split_idx, normalizer = make_loaders(df_feat, normalizer)

print("\n[4/6] Building model …")
model = SolarFlareTCN()
if NUM_GPUS > 1:
    print(f"  Wrapping in nn.DataParallel ({NUM_GPUS} GPUs)")
    model = nn.DataParallel(model)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {n_params:,}")

criterion = AdaptiveTemporalLoss()

print(f"\n[5/6] Training ({EPOCHS} epochs, batch={BATCH_SIZE}×{max(1,NUM_GPUS)}) …")
model, history = train_model(model, tr_loader, vl_loader, criterion, epochs=EPOCHS)

# Save checkpoint
ckpt_path = '/kaggle/working/solar_flare_tcn_best.pt'
torch.save({
    'model_state': model.state_dict(),
    'normalizer_mean': normalizer.mean_,
    'normalizer_std':  normalizer.std_,
    'history': history,
}, ckpt_path)
print(f"  ✓ Checkpoint saved → {ckpt_path}")

print("\n[6/6] Evaluating …")
metrics = evaluate_classification(model, te_loader, normalizer, device)

In [ ]:
n_tr, n_vl, n_te = split_idx
# Rebuild test dataset for plotting
X_all = normalizer.transform(df_feat[FEATURE_COLS].values.astype(np.float32))
y_all = ((df_feat['target'].values.astype(np.float32) - normalizer.mean_[0])
          / normalizer.std_[0])
thrs_norm = normalizer.log1p_thresholds()

te_ds = SolarFlareDataset(X_all[n_tr+n_vl:], y_all[n_tr+n_vl:], thresholds_norm=thrs_norm)
plot_forecast_grid(model, te_ds, normalizer, n_samples=6,
                   title='Test-Set Forecast Grid — 20-min Input → 10-min Prediction (SoLEXS)')

In [ ]:
fig_a = run_custom_inference(
    model, df_feat, normalizer,
    sample_idx=None,        # None → random
    start_time_str=None,    # override with e.g. '2024-01-15 08:30:00'
)

# Find the index with the highest SoLEXS count rate
peak_idx = int(df_raw['solexs_rate'].values.argmax())
peak_idx  = max(0, peak_idx - HISTORY_STEPS)   # start the input window before the flare
print(f"\nLargest flare in dataset at step {peak_idx} "
      f"({df_feat.index[peak_idx] if peak_idx < len(df_feat) else 'OOB'})")

if peak_idx + HISTORY_STEPS + FORECAST_STEPS < len(df_feat):
    fig_b = run_custom_inference(
        model, df_feat, normalizer,
        sample_idx=peak_idx,
    )

In [ ]:
def load_checkpoint(ckpt_path='/kaggle/working/solar_flare_tcn_best.pt'):
    ckpt = torch.load(ckpt_path, map_location='cpu')

    _model = SolarFlareTCN()
    state  = {k.replace('module.', ''): v for k, v in ckpt['model_state'].items()}
    _model.load_state_dict(state)
    if NUM_GPUS > 1:
        _model = nn.DataParallel(_model)
    _model = _model.to(device)
    _model.eval()

    _norm = FeatureNormalizer()
    _norm.mean_ = ckpt['normalizer_mean']
    _norm.std_  = ckpt['normalizer_std']

    print(f"✓ Checkpoint loaded from {ckpt_path}")
    return _model, _norm

# Uncomment to load:
model, normalizer = load_checkpoint()